In [1]:
import sys
import os

sys.path.append(os.path.abspath("../.."))

from config.spark import spark


In [2]:
from pyspark.sql.types import StructType,StructField,StringType,IntegerType,DoubleType

geolocation_schema = StructType([
    StructField("geolocation_zip_code_prefix",IntegerType()),
    StructField("geolocation_lat",DoubleType()),
    StructField("geolocation_lng",DoubleType()),
    StructField("geolocation_city",StringType()),
    StructField("geolocation_state",StringType())
])


In [3]:
geolocation_df = (
    spark.read.format('csv')
    .option('header','True')
  #  .option('inferschema','True')
    .schema(geolocation_schema)
    .load('../../landing/files/olist_geolocation_dataset.csv')
    
)

In [4]:
from pyspark.sql.functions import current_timestamp

geolocation_df = geolocation_df.withColumn("ingestion_time",current_timestamp())

In [5]:
geolocation_df.printSchema()

root
 |-- geolocation_zip_code_prefix: integer (nullable = true)
 |-- geolocation_lat: double (nullable = true)
 |-- geolocation_lng: double (nullable = true)
 |-- geolocation_city: string (nullable = true)
 |-- geolocation_state: string (nullable = true)
 |-- ingestion_time: timestamp (nullable = false)



In [6]:
geolocation_df.write.mode("overwrite").saveAsTable("bronze.geolocation")

In [7]:
spark.sql(
    """
    select * from bronze.geolocation
    """
).show(100)

+---------------------------+-------------------+-------------------+----------------+-----------------+--------------------+
|geolocation_zip_code_prefix|    geolocation_lat|    geolocation_lng|geolocation_city|geolocation_state|      ingestion_time|
+---------------------------+-------------------+-------------------+----------------+-----------------+--------------------+
|                      40260|-12.974667518374174| -38.49278440886833|        salvador|               BA|2026-10-19 15:27:...|
|                      40290|-12.993439446058945| -38.49217486892705|        salvador|               BA|2026-10-19 15:27:...|
|                      40231| -12.99697810785041| -38.51486014380056|        salvador|               BA|2026-10-19 15:27:...|
|                      40296| -12.98915578496331|-38.478782694152216|        salvador|               BA|2026-10-19 15:27:...|
|                      40255|-12.973983547435145| -38.49981646373282|        salvador|               BA|2026-10-19 15: